# 07 — Kiểm toán QC và phân đoạn QRS/T trên toàn bộ PTB VCG

Notebook kiểm tra toàn bộ 448 recordings HC/MI đã trích xuất trong notebook 06.
Câu hỏi là QC đang cảnh báo điều gì, biên nào cần xem, và features nhạy với
dịch biên ra sao. **Giữ nguyên V1, nhãn, split và ba CSV gốc; không train ML.**

Chọn kernel **ipcv** rồi **Run All**. Các ô đã được thực thi và lưu kết quả.
Đầu ra riêng: `outputs/vcg_qc_audit_v1/`. Dùng artifacts/checkpoints đã có;
không chạy lại V1 khi median, velocity, magnitude và tiếp tuyến đã được lưu.

**Mức bằng chứng:** tạo được features; qua QC; biên đúng theo tham chiếu là
ba vấn đề khác nhau. Screens phụ chỉ lập ưu tiên review, không tạo ground truth.
Các kết luận số được sinh sau code, không gán sẵn vào Markdown.

## Chuẩn bị dữ liệu trên máy này
Chạy cell dưới một lần mỗi phiên. Dữ liệu PTB được tải từ GitHub Releases nếu còn thiếu.
Notebook 06–09 tự tạo các đầu vào phụ thuộc còn thiếu; lần đầu có thể chạy lâu.
Hướng dẫn và dung lượng: xem `README.md` và `docs/REPRODUCIBILITY.md`.

In [ ]:
from project_setup import prepare_notebook
prepare_notebook(7)

## 1. Import, cấu hình và bảo toàn baseline

**Mục đích:** Khóa V1 và nguồn dữ liệu trước khi kiểm toán.

**Dữ liệu vào:** Các module V1, CSV notebook 06 và artifacts của full evaluation.

**Cách kiểm tra:** Dùng đúng Config đã lưu; ghi hash nguồn và input trước khi đọc/ghi đầu ra audit.

In [ ]:
from pathlib import Path
import sys, json, subprocess, hashlib
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, Image, IFrame
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'vcg_pipeline.py').is_file())
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT/'scripts'))
from vcg_preprocessing import Config
from vcg_pipeline import FEATURE_NAMES
from vcg_qc_audit import (load_inputs, initialize_audit, qc_tables, boundary_audits,
    failure_analysis, create_review_template, review_references, technical_plots,
    prepare_blinded_review, representative_examples, sensitivity_plots,
    extra_analysis_tables, write_audit_report, split_flags, sha256, write_json)
from vcg_qc_rule_inventory import qc_rules_inventory, source_audit_findings
from vcg_feature_boundary_sensitivity import run_feature_sensitivity, summarize_sensitivity
from vcg_qc_audit_gallery import validate_audit_gallery

OUT = ROOT/'outputs/vcg_qc_audit_v1'
frame, manifest, source_config, INPUT, EVALUATION = load_inputs()
CFG = Config(**source_config['config'])
ARTIFACTS = initialize_audit(frame, INPUT, EVALUATION, OUT, source_config)
before_hashes = {str(p): sha256(p) for p in [
    ROOT/'vcg_pipeline.py', ROOT/'vcg_preprocessing.py',
    ROOT/'scripts/run_full_vcg_evaluation.py',
    ROOT/'05_QRS_T_Loops_Step_by_Step.ipynb',
    ROOT/'06_PTB_VCG_12_Features_Train_Valid_Test.ipynb',
    *[INPUT/name for name in ['train_features.csv','valid_features.csv','test_features.csv','split_manifest.csv','config.json']],
]}
pd.set_option('display.max_colwidth', 150)
pd.set_option('display.max_rows', 60)
print('V1 giữ nguyên; output audit:', OUT)
print('Đã đọc', len(frame), 'recordings từ ba CSV notebook 06.')

## 2. Xác minh cohort, schema và hệ thời gian

**Mục đích:** Bảo đảm đúng 448 record duy nhất, không rò rỉ bệnh nhân.

**Dữ liệu vào:** Ba tập train/valid/test, manifest, label, 12 features, boundaries và QC.

**Cách kiểm tra:** Đối chiếu ID/label/split và số features hữu hạn. Mốc là median_relative_ms.

In [ ]:
assert len(frame) == 448 and frame.record_id.is_unique
assert frame.patient_id.nunique() == 200
assert frame.groupby('patient_id').split.nunique().eq(1).all()
display(frame.groupby(['split','label_name']).agg(
    recordings=('record_id','size'), patients=('patient_id','nunique'), full12=('extraction_success','sum')))
display(frame[['record_id','label_name','split','qrs_on_ms','qrs_off_ms','t_on_ms','t_peak_ms','t_off_ms','feature_completeness']].head())
display(Markdown('**Nhận xét:** QRS-on âm là thời điểm trước anchor trong median beat; không phải biên âm của recording. Nhãn HC/MI lấy từ metadata gốc.'))

## 3. Kiểm toán toàn bộ quy tắc QC V1

**Mục đích:** Truy từng flag về điều kiện toán học thực sự.

**Dữ liệu vào:** Source V1, batch và Config.

**Cách kiểm tra:** Phân biệt cờ ảnh hưởng trực tiếp status với input diagnostics/lý do loại beat/cờ bổ sung.

In [ ]:
rules = qc_rules_inventory(CFG)
rules.to_csv(OUT/'qc_rules_inventory.csv', index=False)
findings = source_audit_findings()
write_json(OUT/'source_audit_findings.json', findings)
display(rules[['qc_flag','scope','trigger_condition','threshold_v1','meaning','false_alarm_risk','contributes_to_v1_status']])
display(Markdown(f'**Nhận xét:** Kiểm kê {len(rules)} quy tắc/chẩn đoán; cột scope xác định nguồn. Các ngưỡng heuristic chưa được hiệu chuẩn độc lập. Không diễn giải chỉ dựa tên flag.'))

## 4. Logic needs_review, eligibility và completeness

**Mục đích:** Giải thích vì sao đủ features vẫn có cảnh báo.

**Dữ liệu vào:** qc_status, qc_flags, eligible_for_features, extraction_success.

**Cách kiểm tra:** V1 gộp detector + boundaries + representative warnings; bất kỳ flag nào cũng làm needs_review.

In [ ]:
display(pd.crosstab(frame.qc_status, frame.extraction_success).rename(columns={False:'thiếu_features',True:'đủ_12'}))
display(pd.crosstab(frame.qc_status, frame.eligible_for_features))
display(Markdown('**Nhận xét:** eligible_for_features giữ logic QC V1; extract_features vẫn tính candidate khi có cờ. Cờ input-quality được lưu riêng, không nhập thẳng vào status. failed cần thống kê riêng vì pipeline chưa hoàn tất.'))

## 5. Thống kê QC, đồng xuất hiện và feature completeness

**Mục đích:** Đếm tất cả flags và xác định nhóm đóng góp lớn.

**Dữ liệu vào:** Toàn bộ 448 records, kể cả failed và thiếu features.

**Cách kiểm tra:** Mẫu số theo nhãn HC/MI; một record có thể xuất hiện ở nhiều flags.

In [ ]:
qc = qc_tables(frame, OUT)
assert set(qc['frequency'].qc_flag).issubset(set(rules.qc_flag))
display(pd.DataFrame([qc['totals']]))
display(qc['by_label'][qc['by_label'].split.eq('all')])
display(qc['frequency'])
technical_plots(frame, qc, None, None, OUT)
for name in ['qc_flag_counts.png','qc_flag_rates_by_label.png','qc_cooccurrence.png','flags_and_feature_completeness.png']:
    display(Image(filename=str(OUT/'plots/summary'/name), width=1050))

## 6. Flags phổ biến và nguyên nhân số lượng review cao

**Mục đích:** Phân biệt một cờ phổ biến với nhiều cờ đồng thời.

**Dữ liệu vào:** Frequency, số flags/record, source rule inventory.

**Cách kiểm tra:** Đếm exclusive/overlap; chưa dùng tần suất để kết luận QC false alarm.

In [ ]:
st_only = frame.qc_flags.eq('st_level_requires_review')
st_present = frame.qc_flags.map(lambda text:'st_level_requires_review' in split_flags(text))
display(frame.assign(st_only=st_only, st_present=st_present).groupby('label_name').agg(
    records=('record_id','size'), st_flags=('st_present','sum'), only_st=('st_only','sum')))
display(Markdown(f"**Nhận xét:** {int(st_present.sum())} ca có cờ ST; {int(st_only.sum())} ca chỉ có cờ đó. needs_review={qc['totals']['needs_review']}; cơ chế any-flag khiến các cảnh báo hình thái/dữ liệu đều ảnh hưởng status. Chưa biết bao nhiêu cờ là false alarm."))

## 7. Kiểm tra QRS delineation trên toàn bộ cohort

**Mục đích:** Tái lấy velocity, threshold, anchor, biên và các tín hiệu cần review.

**Dữ liệu vào:** Saved median XYZ/velocity/JSON và biên trong CSV.

**Cách kiểm tra:** Xác minh hệ thời gian; kiểm tra biên, duration/search margins và hoạt động phụ. Hoạt động phụ có thể là P/ST, không tự gọi QRS bị cắt mất.

In [ ]:
qrs, twave, audit_index = boundary_audits(frame, ARTIFACTS, OUT, CFG)
display(qrs.groupby('label_name').agg(
    records=('record_id','size'), qrs_available=('available','sum'),
    duration_median=('qrs_duration_ms','median'), duration_max=('qrs_duration_ms','max'),
    secondary_activity=('secondary_activity_count',lambda s:s.fillna(0).gt(0).sum())))
display(qrs[['record_id','qrs_on_ms','qrs_off_ms','qrs_duration_ms','velocity_peak_ms','qrs_threshold_mV_s','left_search_margin_ms','right_search_margin_ms','screening_signals']].head())
display(Markdown('**Nhận xét:** QRS rộng/nhiều đỉnh không tự bị gán sai. Các ca không có median không có biên để kiểm tra, dù detector vẫn trả anchors. Không có QRS-on/off MAE khi thiếu chuẩn.'))

## 8. Kiểm tra T-wave: contacts, tiếp tuyến và biên

**Mục đích:** Kiểm tra T-on/T-peak/T-off, mức ST/TP và các điểm đặt tiếp tuyến.

**Dữ liệu vào:** Saved magnitude, derivative, search window và contacts.

**Cách kiểm tra:** Khôi phục đúng tín hiệu V1 và kiểm tra candidate; các screens phụ không là nhãn đúng/sai.

In [ ]:
display(twave.groupby('label_name').agg(records=('record_id','size'),
    full_T=('available','sum'), peak_found=('t_peak_ms','count'),
    t_duration_median=('t_duration_ms','median'), t_duration_min=('t_duration_ms','min')))
display(twave[['record_id','t_on_ms','t_peak_ms','t_off_ms','t_duration_ms','rise_contact_ms','rise_slope_mV_s','fall_contact_ms','fall_slope_mV_s','st_level','tp_level','screening_signals']].head())
technical_plots(frame, qc, qrs, twave, OUT)
display(Markdown('**Nhận xét:** Thứ tự/duration hợp lệ chỉ là kiểm tra nội bộ; chưa xác nhận T-peak nằm đúng thành phần sinh lý.'))

## 9. Phân tích riêng st_level_requires_review

**Mục đích:** Làm rõ công thức và khả năng lẫn hình thái với lỗi baseline/biên.

**Dữ liệu vào:** ST/PQ magnitude, noise PQ, T amplitude và T-on.

**Cách kiểm tra:** Tái tính ST−PQ > max(5×noise, 0.15×amplitude). So counterfactual giao điểm PQ chỉ để đo ảnh hưởng reference.

In [ ]:
measurable = twave.st_minus_pq_mV.notna()
assert twave.loc[measurable,'st_flag_recomputed'].eq(twave.loc[measurable,'st_flag_present']).all()
display(twave.groupby('label_name')[['st_minus_pq_mV','st_delta_peak_fraction','st_to_pq_intercept_shift_ms']].agg(['count','median','max']))
display(pd.crosstab(twave.st_flag_present.fillna(False), twave.t_threshold_on_ms.notna(),
    rownames=['ST_flag'], colnames=['threshold_on_available']))
display(Image(filename=str(OUT/'plots/summary/st_rule_and_t_on.png'), width=1150))
display(Markdown('**Nhận xét:** Đây là magnitude không dấu, không phải chẩn đoán ECG ST elevation/depression. ST thật khác baseline và PQ chọn chưa phù hợp đều là giả thuyết; cần tham chiếu độc lập. Counterfactual không thay thế biên V1 hoặc tạo ground truth.'))

## 10. T-end disagreement, thiếu baseline và lobe

**Mục đích:** Kiểm tra khả năng giải các phương pháp và độ bất đồng.

**Dữ liệu vào:** T tangent/threshold candidates và nguồn lobe peak.

**Cách kiểm tra:** Báo cả n có hai phương pháp và n thiếu; gap thiếu không điền 0.

In [ ]:
method_rows = []
for label, group in twave.groupby('label_name'):
    for side in ['on','off']:
        values = group[f't_{side}_method_gap_ms'].dropna()
        method_rows.append(dict(label=label,boundary='T-'+side,all_records=len(group),both_methods=len(values),
            median_ms=values.median(),max_ms=values.max(),gt100ms=int(values.gt(100).sum())))
display(pd.DataFrame(method_rows))
display(twave[twave.n_significant_lobes.fillna(0).gt(1)][['record_id','split','n_significant_lobes','significant_lobes_outside_valid_window','candidate_peaks_json']])
display(Image(filename=str(OUT/'plots/summary/t_method_disagreement.png'), width=1100))
display(Markdown('**Nhận xét:** Method agreement chưa phải accuracy. Caveat source: significant lobes không áp lại peak-valid mask; bảng trên kiểm tra có kích hoạt trên cohort hay không, không tự sửa code.'))

## 11. Kiểm tra toàn bộ records không có QC flags

**Mục đích:** Tìm những ca QC có thể bỏ qua, không chỉ xem nhóm cảnh báo.

**Dữ liệu vào:** Tất cả records n_flags=0.

**Cách kiểm tra:** Screens được khai báo trước trong audit_config; bất đồng >60 ms chỉ lập candidate nhóm C.

In [ ]:
unflagged = audit_index[audit_index.n_flags.eq(0)]
candidates_C = unflagged[unflagged.screen_positive]
display(candidates_C[['record_id','label_name','split','t_on_method_gap_ms','t_off_method_gap_ms','suspicion_indicators']])
display(Markdown(f'**Nhận xét:** Đã screen {len(unflagged)} ca không cờ; {len(candidates_C)} có dấu hiệu phụ cần review. Không gọi đó là {len(candidates_C)} missed errors. Số ngoài needs_review gồm cả failed, không phải tất cả đều qua QC.'))

## 12. Truy nguyên bảy records thiếu features

**Mục đích:** Tách detector, giữ nhịp, median và T delineation.

**Dữ liệu vào:** Input-quality, beat QC, exception trace và features từng record.

**Cách kiểm tra:** Đọc chẩn đoán đã lưu ở đúng bước; giữ NaN, không tạo nhịp/biên giả.

In [ ]:
failures = failure_analysis(frame, ARTIFACTS, OUT)
display(failures[['record_id','split','feature_completeness','detected_anchors','retained_beats','excluded_beats','beat_reasons_json','qrs_delineation','t_delineation','feature_extraction']])
display(Markdown(f'**Nhận xét:** {len(failures)} record thiếu features. Các ca có anchors nhưng thiếu median cần review detector/beat-selection trên tín hiệu gốc; các ca đủ QRS nhưng thiếu T giữ được 7 QRS features. Không sửa V1 trong audit.'))

## 13. Gallery tất cả records và xem từng ca

**Mục đích:** Cho phép kiểm tra trực quan cả ca có/không cờ và lỗi kỹ thuật.

**Dữ liệu vào:** Artifacts thật; raw XYZ chỉ đọc lại ở ca không có median.

**Cách kiểm tra:** Hình A–D có XYZ, raw/smoothed velocity, magnitude/contacts/tangents và loops. Filters theo label/flag/split/status.

In [ ]:
command = [sys.executable, str(ROOT/'scripts/run_vcg_qc_audit_stage.py'), 'gallery', '--output-dir', str(OUT), '--workers', '4']
completed = subprocess.run(command, cwd=ROOT, check=True, capture_output=True, text=True)
print(completed.stdout.strip())
gallery_check = validate_audit_gallery(OUT, set(frame.record_id))
display(pd.DataFrame([gallery_check]))
examples = representative_examples(audit_index, OUT)
display(examples[['group','record_id','label_name','split','qc_status','feature_completeness']])
gallery_manifest = pd.read_csv(OUT/'gallery_manifest.csv').set_index('record_id')
def show_record(record_id):
    row = gallery_manifest.loc[record_id]
    display(Image(filename=str(OUT/row['plot_path']), width=1200))
RECORD_TO_INSPECT = 'patient034/s0109lre'  # sửa ID này để xem record khác
show_record(RECORD_TO_INSPECT)
show_record('patient051/s0179lre')
show_record('patient104/s0306lre')
display(Markdown('[Mở gallery đủ 448 ca](outputs/vcg_qc_audit_v1/qc_review_gallery.html). Các ví dụ gồm ca thiếu T, T rất ngắn và ca không cờ; chưa phải kết luận chuyên gia.'))

## 14. Tham chiếu độc lập hoặc manual review

**Mục đích:** Chuẩn bị mốc tham chiếu độc lập và bảo vệ khỏi nhãn đúng/sai giả.

**Dữ liệu vào:** Mẫu train/valid có phân tầng, ít nhất 30 HC và 30 MI, gồm có/không cờ.

**Cách kiểm tra:** Lượt đầu dùng blinded CSV/tín hiệu không overlay. Chỉ nhận confirmed + reviewer + independent_reference=true.

In [ ]:
sample = create_review_template(audit_index, OUT)
blinded = prepare_blinded_review(OUT, ARTIFACTS, ROOT/'content/ptbdb/physionet.org/files/ptbdb/1.0.0', CFG)
display(sample.groupby(['label_name','split']).agg(records=('record_id','size'),patients=('patient_id','nunique')))
assert sample['split'].isin(['train','valid']).all()
assert sample.groupby('label_name').size().ge(30).all()
REVIEW_FILE = OUT/'manual_review_template.csv'  # có thể đổi sang manual_review_blinded.csv sau chú giải
REFERENCE_ACCEPTANCE_MS = None  # chỉ đặt sau khi khai báo tiêu chí; không mặc định chuẩn lâm sàng
comparison, reference_summary, reference_audit = review_references(
    frame, OUT, EVALUATION, acceptance_ms=REFERENCE_ACCEPTANCE_MS, review_path=REVIEW_FILE)
display(reference_audit.accepted.value_counts().rename('reference_rows'))
display(comparison.head())
display(pd.read_csv(OUT/'qc_reference_rates.csv'))
display(Markdown('**Nhận xét:** Khi chưa có chú giải hợp lệ, comparison rỗng và false-alarm/missed-error rate để NaN. Challenge 2006 QT lead II và PTB-XL+ không tự trở thành chuẩn bốn biên Frank median của các record này.'))

## 15. Độ nhạy 12 features với dịch từng biên

**Mục đích:** Đo ảnh hưởng kỹ thuật của sai lệch biên giả định.

**Dữ liệu vào:** Median XYZ, biên và full-median SG speed; tất cả 448 records.

**Cách kiểm tra:** Dịch một biên −20/−10/+10/+20 ms. Tính lại công thức V1; reject thứ tự/loop không hợp lệ, không clamp. ε=1e−9 trong đơn vị từng feature.

In [ ]:
sensitivity = run_feature_sensitivity(frame, ARTIFACTS, OUT, cfg=CFG, resume=True)
full12_long = sensitivity['long'][sensitivity['long'].original_feature_completeness.eq(12)]
full12_summary, full12_rank = summarize_sensitivity(full12_long)
full12_summary.to_csv(OUT/'feature_sensitivity_full12_summary.csv', index=False)
full12_rank.to_csv(OUT/'feature_sensitivity_full12_rank.csv', index=False)
sensitivity_plots(sensitivity, OUT)
display(pd.DataFrame([{key:sensitivity['metadata'][key] for key in ['n_records','n_scenarios','n_long_rows','cache_hits']}]))
primary = full12_rank[full12_rank.scope.eq('train_valid') & full12_rank.grouping.eq('all')].copy()
primary['relative_median_pct'] = primary.relative_delta_median*100
primary['relative_p95_pct'] = primary.relative_delta_p95*100
display(primary[['feature','unit','absolute_delta_median','absolute_delta_p95','relative_median_pct','relative_p95_pct','n_valid','n_eligible_but_invalid']])
display(Image(filename=str(OUT/'plots/summary/feature_sensitivity_heatmap.png'), width=1150))
display(Markdown('**Nhận xét:** Rank chỉ gộp biên của chính loop, dùng train/valid để định hướng. Relative change lớn khi feature gần zero cần đọc cùng absolute change. Invalid/unavailable scenarios không biến thành Δ=0. Đây không phải sai số biên thực hay feature importance của classifier.'))

## 16. Tổng hợp bằng chứng và khác biệt HC/MI

**Mục đích:** Tách các kết quả quan sát khỏi phán quyết tham chiếu.

**Dữ liệu vào:** Audit index, ST strata, method gaps và sensitivity theo patient.

**Cách kiểm tra:** A chưa được xác nhận; B/C chỉ là candidates. Các ca thành công vẫn D_chưa xác minh nếu không có chuẩn.

In [ ]:
extra = extra_analysis_tables(OUT)
display(audit_index.groupby(['label_name','screening_route']).size().rename('records'))
display(audit_index.groupby(['label_name','audit_group']).size().rename('records'))
display(extra['method_agreement'])
display(extra['st_strata'])
label_sensitivity = full12_rank[full12_rank.scope.eq('train_valid') & full12_rank.grouping.eq('label')]
display(label_sensitivity[['label_name','feature','absolute_delta_median','relative_delta_median','relative_delta_p95']])
display(Markdown('**Nhận xét:** Khác biệt HC/MI ở đây là mô tả. Các recordings cùng patient phụ thuộc nhau; bảng patient medians được xuất riêng. ST strata chưa có confirmed errors, nên chưa thể xác định mức ST nào bắt đầu gây lỗi.'))

## 17. Độ tin cậy V1, source caveats và 11 câu trả lời

**Mục đích:** Trả lời từng câu bằng mức bằng chứng phù hợp.

**Dữ liệu vào:** Kết quả audit đã tính, không hardcode số kết luận trước chạy.

**Cách kiểm tra:** Tách annotation-confirmed / technical evidence / hypothesis. Chỉ đề xuất hướng V2, không thay V1.

In [ ]:
display(pd.DataFrame(source_audit_findings()))
answers = write_audit_report(OUT)
for _, row in answers.iterrows():
    display(Markdown('### '+row.question+'\n\n**Mức bằng chứng:** '+row.evidence_level+'.\n\n'+row.answer))

## 18. Kiểm tra output và khả năng chạy lại

**Mục đích:** Bảo đảm đủ coverage, không ghi đè nguồn và có cache an toàn.

**Dữ liệu vào:** Toàn bộ CSV, gallery, source/input hashes và checkpoints.

**Cách kiểm tra:** So hash trước/sau. Khi Run All lần nữa: boundary audit được tái kiểm, sensitivity/gallery dùng cache khớp; review CSV đã có không bị ghi đè.

In [ ]:
assert all(sha256(Path(path)) == digest for path,digest in before_hashes.items())
assert len(qrs) == len(twave) == len(frame) == 448
assert len(sensitivity['scenarios']) == len(frame)*4*4
assert len(sensitivity['long']) == len(frame)*4*4*len(FEATURE_NAMES)
assert sensitivity['baseline_check'].matches_csv.all()
required = ['qc_flag_frequency.csv','qc_by_label.csv','qc_rules_inventory.csv',
    'qrs_boundary_audit.csv','t_boundary_audit.csv','feature_sensitivity.csv',
    'suspicious_records.csv','failed_records_analysis.csv','review_priority.csv',
    'manual_review_template.csv','reference_comparison.csv','qc_review_gallery.html',
    'audit_config.json','qc_audit_report.md']
assert all((OUT/name).is_file() for name in required)
display(pd.DataFrame({'artifact':required,'exists':[True]*len(required)}))
write_json(OUT/'notebook_execution_checks.json', {
    'records':len(frame),'patients':frame.patient_id.nunique(),
    'baseline_and_notebook06_inputs_unchanged':True,
    'sensitivity_scenarios':len(sensitivity['scenarios']),
    'sensitivity_rows':len(sensitivity['long']),
    'expert_rows_accepted':int(reference_audit.accepted.sum()),
    'test_used_for_tuning':False,'baseline_modified':False,
})
display(Markdown('**Hoàn tất:** [Báo cáo 11 câu hỏi](outputs/vcg_qc_audit_v1/qc_audit_report.md) · [Gallery 448 ca](outputs/vcg_qc_audit_v1/qc_review_gallery.html) · [Mẫu review độc lập](outputs/vcg_qc_audit_v1/manual_review_blinded.csv). Chưa đo accuracy biên khi chưa có annotation phù hợp.'))